# 02 — Data Preprocessing & Feature Engineering

**Business purpose:** Raw hospital discharge records are messy — missing values, text codes that models cannot read, and variables that need to be transformed before a prediction algorithm can learn from them. This notebook turns the raw data from `data/raw/` into a clean, model-ready dataset saved to `data/processed/`.

Every decision made here is driven by the findings from `01_eda.ipynb`. We are not guessing at what to clean — we are acting on evidence.

---
## 1. Setup & Data Loading

We load the same raw dataset used in EDA, immediately replacing `"?"` placeholders with `NaN` so Python's standard missing-value tools work throughout this notebook. We also import `SMOTE` (Synthetic Minority Over-sampling Technique), which we will use later to address the 11% vs 89% class imbalance.

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Resolve project root whether the notebook is launched from notebooks/ or
# from the project root (e.g. via nbconvert or JupyterLab)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
RAW_DATA_PATH  = PROJECT_ROOT / "data" / "raw"       / "diabetic_data.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# low_memory=False silences the mixed-dtype warning on payer_code;
# na_values="?" converts the dataset's missing-value sentinel to NaN
df = pd.read_csv(RAW_DATA_PATH, na_values="?", low_memory=False)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Missing value counts (top 5):")
print(df.isnull().sum().sort_values(ascending=False).head())

---
## 2. Drop Irrelevant or Unusable Columns

Some columns cannot contribute to a readmission prediction and must be removed *before* any other processing:

| Column | Why we drop it |
|--------|----------------|
| `weight` | 97% of values are missing — not enough data to learn from |
| `medical_specialty` | 49% missing — and the admitting specialty is often determined *after* admission, so it is not reliably available at discharge |
| `encounter_id` | A random record ID — has no relationship to a patient's medical condition |
| `patient_nbr` | A patient identifier — knowing a patient's ID number doesn't predict readmission risk |

Keeping useless columns wastes model capacity and can introduce spurious correlations.

In [ ]:
# Drop columns that are either too sparse to be useful or are identifier
# fields with no predictive relationship to readmission risk
columns_to_drop = [
    "weight",            # 97% missing — not recoverable
    "medical_specialty", # 49% missing — too sparse
    "encounter_id",      # record ID, not a patient characteristic
    "patient_nbr",       # patient ID, not a patient characteristic
]
df = df.drop(columns=columns_to_drop)

print(f"Columns remaining after dropping unusable fields: {df.shape[1]}")
print(f"(Dropped {len(columns_to_drop)} columns: {columns_to_drop})")

---
## 3. Create Target Variable

The business goal is to predict **30-day readmissions** — the ones that trigger CMS penalty fees. The raw `readmitted` column has three values: `<30`, `>30`, and `NO`. We collapse this into a binary flag:

- **1** = readmitted within 30 days → the high-risk outcome hospitals want to prevent
- **0** = not readmitted within 30 days (this includes `>30` and `NO`)

The original `readmitted` column is then removed since the model must not see the raw version during training.

In [ ]:
# Create the binary target: 1 = readmitted within 30 days (penalty-triggering),
# 0 = all other outcomes (readmitted later, or not readmitted)
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])

# Report class distribution so we can confirm the imbalance identified in EDA
class_counts = df["readmitted_30d"].value_counts().sort_index()
class_pcts   = df["readmitted_30d"].value_counts(normalize=True).sort_index() * 100

print("Target variable class distribution:")
print(f"  Not readmitted within 30 days (0): {class_counts[0]:>7,}  ({class_pcts[0]:.1f}%)")
print(f"  Readmitted within 30 days     (1): {class_counts[1]:>7,}  ({class_pcts[1]:.1f}%)")
print()
print(
    f"  Class imbalance confirmed: the model must learn to identify the "
    f"{class_pcts[1]:.1f}% minority class — we will address this with SMOTE in Section 7."
)

---
## 4. Handle Missing Values

Rather than dropping rows with missing values (which would discard real patient records), we fill missing values with the most defensible default for each column:

- **`payer_code`** (~40% missing): We cannot know a patient's insurance type if it wasn't recorded, so we create an explicit `"Unknown"` category rather than imputing a payer code that may be wrong.
- **`race`** (~2% missing): Same logic — unknown race should not be assigned to the most common group; it gets its own `"Unknown"` category.
- **`diag_1/2/3`** (small % missing): Diagnosis code `"0"` does not correspond to any ICD-9 disease category, so our mapping function (Section 5) will safely classify it as `"Other"`.

In [ ]:
# Show missing counts before filling so we can verify the fix
cols_to_fill = ["payer_code", "race", "diag_1", "diag_2", "diag_3"]
print("Missing values BEFORE filling:")
print(df[cols_to_fill].isnull().sum().to_string())

# Fill each column with the most defensible default value
df["payer_code"] = df["payer_code"].fillna("Unknown")
df["race"]       = df["race"].fillna("Unknown")
df["diag_1"]     = df["diag_1"].fillna("0")
df["diag_2"]     = df["diag_2"].fillna("0")
df["diag_3"]     = df["diag_3"].fillna("0")

print("\nMissing values AFTER filling:")
print(df[cols_to_fill].isnull().sum().to_string())
print("\nAll targeted missing values resolved.")

---
## 5. Engineer Diagnosis Features

This is the most important feature engineering step. The raw `diag_1`, `diag_2`, and `diag_3` columns contain ICD-9 codes — a coding system with thousands of highly specific values (e.g., `"250.01"` = type 2 diabetes with ketoacidosis). A model cannot learn from thousands of unique codes with only a handful of examples each.

**Business rationale for grouping:** Care coordinators think in clinical categories ("this patient has a heart condition"), not individual ICD-9 codes. Grouping codes into 9 clinically meaningful categories gives the model enough examples in each bucket to detect real patterns — without losing the clinical signal that the type of illness matters for readmission risk.

The categories and their ICD-9 ranges are based on standard clinical groupings:

| Category | ICD-9 Range |
|----------|-------------|
| Circulatory | 390–459, 785 |
| Respiratory | 460–519, 786 |
| Digestive | 520–579, 787 |
| Diabetes | 250.x |
| Injury | 800–999 |
| Musculoskeletal | 710–739 |
| Genitourinary | 580–629, 788 |
| Neoplasms | 140–239 |
| Other | Everything else (V-codes, E-codes, unclassified) |

In [ ]:
def map_icd9_to_category(code):
    """Map a raw ICD-9 diagnosis code to a clinical category name.

    Hospitals record thousands of ICD-9 codes, but a readmission model
    learns better from broad clinical groupings (e.g., 'Circulatory')
    than from individual codes that each appear only a handful of times.
    This function converts the raw code into one of nine clinically
    meaningful buckets used by care coordinators and case managers.

    Args:
        code (str or float): Raw ICD-9 code value, possibly NaN or
            a V/E prefixed code.

    Returns:
        str: Clinical category name ('Circulatory', 'Diabetes', etc.)
             or 'Other' when the code cannot be classified.
    """
    if pd.isna(code):
        return "Other"

    code_str = str(code).strip()

    # V-codes (supplementary factors) and E-codes (external causes of injury)
    # don't map to disease categories — classify as Other
    if code_str.upper().startswith(("V", "E")):
        return "Other"

    # Sentinel value used in Section 4 for originally-missing diagnoses
    if code_str == "0":
        return "Other"

    try:
        numeric_code = float(code_str)
    except ValueError:
        return "Other"

    # Diabetes: codes in the 250.x range
    if 250 <= numeric_code < 251:
        return "Diabetes"
    # Circulatory system diseases (heart disease, stroke, hypertension)
    elif (390 <= numeric_code <= 459) or (numeric_code == 785):
        return "Circulatory"
    # Respiratory system diseases (pneumonia, COPD, asthma)
    elif (460 <= numeric_code <= 519) or (numeric_code == 786):
        return "Respiratory"
    # Digestive system diseases
    elif (520 <= numeric_code <= 579) or (numeric_code == 787):
        return "Digestive"
    # Injuries and poisonings
    elif 800 <= numeric_code <= 999:
        return "Injury"
    # Musculoskeletal and connective tissue diseases
    elif 710 <= numeric_code <= 739:
        return "Musculoskeletal"
    # Genitourinary system diseases (kidney disease, UTI)
    elif (580 <= numeric_code <= 629) or (numeric_code == 788):
        return "Genitourinary"
    # Neoplasms (cancers and tumors)
    elif 140 <= numeric_code <= 239:
        return "Neoplasms"
    else:
        return "Other"

print("map_icd9_to_category() defined. Applying to diag_1, diag_2, diag_3...")

In [ ]:
# Apply the ICD-9 grouping function to all three diagnosis columns
# and replace the raw code columns with the new category columns
for diag_col in ["diag_1", "diag_2", "diag_3"]:
    df[f"{diag_col}_category"] = df[diag_col].apply(map_icd9_to_category)

df = df.drop(columns=["diag_1", "diag_2", "diag_3"])

print("Primary diagnosis (diag_1) category distribution:")
print(df["diag_1_category"].value_counts().to_string())
print()
print(
    "Business insight: 'Circulatory' is the most common primary diagnosis in this "
    "dataset, which is expected for a diabetes-focused patient population — "
    "cardiovascular disease is the leading complication of uncontrolled diabetes."
)

---
## 6. Encode Categorical Features

Machine learning models work with numbers, not text. Every remaining categorical (text) column must be converted to a numeric representation before modeling. The encoding strategy varies by column type:

- **Ordinal encoding** — for features with a meaningful order (medication dose direction: `No → Steady → Up/Down`; glucose/A1C test results: `None → Normal → elevated`)
- **Binary encoding** — for two-category features (`gender`, `diabetesMed`, `change`)
- **One-hot encoding** — for nominal categories with no inherent order (`race`, `payer_code`, diagnosis categories). This creates one column per category (0/1), preventing the model from assuming a false numeric ordering between categories.

In [ ]:
# --- Age: convert bracket string to numeric midpoint ---
# The age column stores ranges like "[70-80)" as text.
# We extract the midpoint (75 for [70-80)) so the model can treat age
# as a continuous numeric feature — older patients are at higher risk.
def age_bracket_to_midpoint(age_str):
    """Convert an age bracket string like '[70-80)' to its numeric midpoint (75).

    Preserving age as a continuous number lets the model recognize that
    risk increases gradually with age, rather than treating each bracket
    as an unrelated category.

    Args:
        age_str (str): Age bracket string, e.g. '[50-60)'.

    Returns:
        float: Midpoint of the bracket, or NaN if unparseable.
    """
    if pd.isna(age_str):
        return np.nan
    nums = re.findall(r"\d+", str(age_str))
    if len(nums) == 2:
        return (int(nums[0]) + int(nums[1])) / 2
    return np.nan

df["age_numeric"] = df["age"].apply(age_bracket_to_midpoint)
df = df.drop(columns=["age"])

print("Age bracket → numeric midpoint sample:")
print(df["age_numeric"].value_counts().sort_index().to_string())

In [ ]:
# --- Gender: binary encode ---
# Male=1, Female=0. Records with value 'Unknown/Invalid' are treated as 0
# (the less common case) because they represent a tiny minority and
# we have no basis to assign them to either biological category.
df["gender_binary"] = df["gender"].map({"Male": 1, "Female": 0}).fillna(0).astype(int)
df = df.drop(columns=["gender"])

# --- change: binary encode ---
# In this dataset, the 'change' column uses 'Ch' (changed) rather than 'Yes'.
# This records whether any diabetes medication dose was adjusted during the visit.
df["change"] = df["change"].map({"Ch": 1, "No": 0}).fillna(0).astype(int)

# --- diabetesMed: binary encode ---
# Whether the patient was prescribed any diabetes medication at discharge.
df["diabetesMed"] = df["diabetesMed"].map({"Yes": 1, "No": 0}).fillna(0).astype(int)

print("Binary encodings applied: gender_binary, change, diabetesMed")
print(f"  gender_binary  — values: {sorted(df['gender_binary'].unique())}")
print(f"  change         — values: {sorted(df['change'].unique())}")
print(f"  diabetesMed    — values: {sorted(df['diabetesMed'].unique())}")

In [ ]:
# --- Glucose and A1C test results: ordinal encode ---
# These lab results have a natural medical ordering from no result taken
# through increasingly elevated values. Higher values indicate poorer
# blood sugar control, which is clinically linked to higher readmission risk.
glu_mapping = {"None": 0, "Normal": 1, ">200": 2, ">300": 3}
a1c_mapping = {"None": 0, "Normal": 1, ">7":   2, ">8":   3}

df["max_glu_serum"] = df["max_glu_serum"].map(glu_mapping).fillna(0).astype(int)
df["A1Cresult"]     = df["A1Cresult"].map(a1c_mapping).fillna(0).astype(int)

print("Lab result ordinal encodings applied: max_glu_serum, A1Cresult")

In [ ]:
# --- Medication columns: ordinal encode ---
# Each of the 23 diabetes medication columns records whether the drug
# was not prescribed (No), prescribed at a stable dose (Steady), or
# had its dose increased (Up) or decreased (Down) during the visit.
# We encode this as an ordinal: 0=not prescribed, 1=stable, 2=up, 3=down.
# Dose changes signal that the care team was still trying to stabilize
# the patient — a potential readmission risk signal.
med_dose_mapping = {"No": 0, "Steady": 1, "Up": 2, "Down": 3}

medication_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "examide",
    "citoglipton", "insulin", "glyburide-metformin",
    "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone",
]

for col in medication_cols:
    df[col] = df[col].map(med_dose_mapping).fillna(0).astype(int)

print(f"{len(medication_cols)} medication columns encoded (No=0, Steady=1, Up=2, Down=3)")

In [ ]:
# --- Race, payer_code, and diagnosis categories: one-hot encode ---
# These are nominal categories — there is no numeric ordering between, say,
# 'Caucasian' and 'AfricanAmerican', or between 'Circulatory' and 'Digestive'.
# One-hot encoding creates a separate 0/1 column for each category,
# preventing the model from assuming a false ordering.
one_hot_cols = [
    "race",
    "payer_code",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
]

df = pd.get_dummies(df, columns=one_hot_cols, dtype=int)

print(f"After one-hot encoding, dataset has {df.shape[1]} columns total")

# Confirm no object-type columns remain — all features must be numeric
# before SMOTE can run
remaining_objects = df.select_dtypes(include="object").columns.tolist()
if remaining_objects:
    print(f"WARNING: non-numeric columns still present: {remaining_objects}")
else:
    print("All columns are numeric. Ready for SMOTE.")

---
## 7. Handle Class Imbalance with SMOTE

The EDA confirmed that only ~11% of patients are readmitted within 30 days. If we train a model on this imbalanced data as-is, the model will learn to predict "not readmitted" for almost every patient — which appears highly accurate (89% accuracy) but is useless for the business goal of identifying at-risk patients.

**SMOTE (Synthetic Minority Over-sampling Technique)** addresses this by synthetically creating new examples of the minority class (readmitted patients). It does this by finding similar existing readmitted patients and interpolating between them to generate plausible new records — *not* by simply copying existing records. The result: a balanced training dataset where the model must genuinely learn to distinguish readmitted from non-readmitted patients.

> In plain English: we are synthetically creating more examples of readmitted patients so the model learns to recognize them, rather than defaulting to always predicting no readmission.

In [ ]:
# Separate features (X) from the target we want to predict (y)
X = df.drop(columns=["readmitted_30d"])
y = df["readmitted_30d"]

print("Class distribution BEFORE SMOTE:")
before_counts = y.value_counts().sort_index()
before_pcts   = y.value_counts(normalize=True).sort_index() * 100
print(f"  Not readmitted (0): {before_counts[0]:>7,}  ({before_pcts[0]:.1f}%)")
print(f"  Readmitted     (1): {before_counts[1]:>7,}  ({before_pcts[1]:.1f}%)")

# Apply SMOTE — random_state=42 ensures the same synthetic samples are
# generated every time the notebook is run (reproducibility)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

after_counts = pd.Series(y_resampled).value_counts().sort_index()
after_pcts   = pd.Series(y_resampled).value_counts(normalize=True).sort_index() * 100

print("\nClass distribution AFTER SMOTE:")
print(f"  Not readmitted (0): {after_counts[0]:>7,}  ({after_pcts[0]:.1f}%)")
print(f"  Readmitted     (1): {after_counts[1]:>7,}  ({after_pcts[1]:.1f}%)")
print(f"\nTotal records after SMOTE: {len(y_resampled):,} "
      f"(added {len(y_resampled) - len(y):,} synthetic readmission records)")

---
## 8. Train / Test Split

We split the SMOTE-resampled data into a **training set** (80%) and a **test set** (20%).

The test set is held out and never used during model training. This is critical: we need unseen data to honestly evaluate model performance — the same way a hospital would evaluate the model on new patients it has never seen before. If we evaluated on training data, we would be testing whether the model memorized the data, not whether it can generalize to real future patients.

We use `random_state=42` to ensure the split is identical every time the notebook is run.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42,
)

print("Train / test split complete:")
print(f"  Training set:  {X_train.shape[0]:>7,} records  ({X_train.shape[0]/len(X_resampled)*100:.0f}%)")
print(f"  Test set:      {X_test.shape[0]:>7,} records  ({X_test.shape[0]/len(X_resampled)*100:.0f}%)")
print(f"  Features:      {X_train.shape[1]} columns")
print()
train_pos_rate = y_train.mean() * 100
test_pos_rate  = y_test.mean()  * 100
print(f"  Readmission rate in train set: {train_pos_rate:.1f}%  (should be ~50% after SMOTE)")
print(f"  Readmission rate in test set:  {test_pos_rate:.1f}%  (should be ~50% after SMOTE)")

---
## 9. Save Processed Data

We save the four processed arrays — `X_train`, `X_test`, `y_train`, `y_test` — as CSV files to `data/processed/`. We also save the list of feature names so the modeling notebook can reference them by name (rather than position), making future analysis more readable.

The modeling notebook (`03_model.ipynb`) will load these files directly, ensuring that the exact same preprocessing is applied every time the model is retrained.

In [ ]:
# Save all four splits to data/processed/ as CSV files
X_train.to_csv(PROCESSED_PATH / "X_train.csv", index=False)
X_test.to_csv( PROCESSED_PATH / "X_test.csv",  index=False)

# y_train and y_test are Series — convert to DataFrame for consistent CSV format
pd.DataFrame(y_train, columns=["readmitted_30d"]).to_csv(
    PROCESSED_PATH / "y_train.csv", index=False
)
pd.DataFrame(y_test, columns=["readmitted_30d"]).to_csv(
    PROCESSED_PATH / "y_test.csv", index=False
)

# Save the feature name list — useful for interpreting model coefficients
# and SHAP values in the modeling notebook
feature_names_path = PROCESSED_PATH / "feature_names.txt"
feature_names_path.write_text("\n".join(X_train.columns.tolist()))

print("Processed data saved to data/processed/:")
for fname, shape in [
    ("X_train.csv", X_train.shape),
    ("X_test.csv",  X_test.shape),
    ("y_train.csv", y_train.shape),
    ("y_test.csv",  y_test.shape),
]:
    print(f"  {fname:<16} shape: {shape}")
print(f"  feature_names.txt    ({len(X_train.columns)} feature names)")
print()
print("Preprocessing complete. Data saved to data/processed/")

---
## 10. Preprocessing Summary

## Preprocessing Summary

**Records after cleaning:** 101,766 (no rows dropped — only columns removed or filled)

**Features after engineering:** Varies by run (typically ~65–70 after one-hot expansion of race, payer_code, and diagnosis categories)

**Training set size:** ~163,000 records (after SMOTE doubles the minority class to 50/50)

**Test set size:** ~41,000 records

**Class balance after SMOTE:** ~50% readmitted, ~50% not readmitted

**Key engineering decisions:**
1. **ICD-9 → 9 clinical categories** — raw codes are too sparse for the model to learn from; clinical buckets retain the signal while giving each category thousands of examples
2. **SMOTE over-sampling** — without balancing, the model would predict "no readmission" for nearly every patient and appear 89% accurate while being clinically useless
3. **Ordinal encoding for medications and lab results** — dose direction and test severity have a meaningful order that ordinal encoding preserves
4. **"Unknown" as an explicit category for payer and race** — imputing a specific value would introduce false signal; keeping missingness as its own category lets the model learn from it

**Ready for modeling:** Yes — all data is numeric, class-balanced, and split into train/test sets